# RAG Knowledge-Base Document Pipeline

Incrementally processes documents uploaded to the `agentic_ai.01_bronze/raw_documents` UC volume: discovers new/changed files, parses them with the `ai_parse_document` SQL AI function and chunks them with the `ai_prep_search` AI function (semantic, context-enriched chunks), then upserts them into `agentic_ai.03_gold.final_chunk` — the source table of the `agentic_ai.03_gold.document_index` AI Search (Vector Search) index. Embeddings are fully managed by that DELTA_SYNC index; after each run the notebook only triggers an index sync.

Production properties: widget-driven configuration (overridable by a Lakeflow job), idempotent re-runs via a processing-log state table, per-file error isolation (one bad file never fails the run), stale-chunk cleanup on re-uploads, and a run summary. Designed to be scheduled as a Lakeflow job task.

In [0]:
# ------------------------------------------------------------------
# 1. CONFIGURATION — all knobs are notebook widgets so a scheduled
#    Lakeflow job can override any of them via task parameters.
# ------------------------------------------------------------------
import uuid

dbutils.widgets.text("volume_path", "/Volumes/agentic_ai/01_bronze/raw_documents", "Source UC volume path")
dbutils.widgets.text("gold_table", "agentic_ai.03_gold.final_chunk", "Gold chunks Delta table")
dbutils.widgets.text("log_table", "agentic_ai.01_bronze.document_processing_log", "Processing-log state table")
dbutils.widgets.text("index_name", "agentic_ai.03_gold.document_index", "AI Search (Vector Search) index")
dbutils.widgets.text("chunk_size", "4000", "Max characters per chunk")
dbutils.widgets.text("chunk_overlap", "400", "Overlap characters between consecutive chunks")

VOLUME_PATH   = dbutils.widgets.get("volume_path").rstrip("/")
GOLD_TABLE    = dbutils.widgets.get("gold_table")
LOG_TABLE     = dbutils.widgets.get("log_table")
INDEX_NAME    = dbutils.widgets.get("index_name")
CHUNK_SIZE    = int(dbutils.widgets.get("chunk_size"))
CHUNK_OVERLAP = int(dbutils.widgets.get("chunk_overlap"))

assert CHUNK_SIZE > 0, "chunk_size must be positive"
assert 0 <= CHUNK_OVERLAP < CHUNK_SIZE, "chunk_overlap must be smaller than chunk_size"

# Formats the ai_parse_document SQL AI function handles natively
AI_PARSE_EXTENSIONS     = {"pdf", "jpg", "jpeg", "png", "tiff", "tif", "doc", "docx", "ppt", "pptx"}
# Formats parsed natively in Python (driver-side, per file)
TEXT_EXTENSIONS        = {"txt", "md"}
SPREADSHEET_EXTENSIONS  = {"xlsx", "xls"}
SUPPORTED_EXTENSIONS    = AI_PARSE_EXTENSIONS | TEXT_EXTENSIONS | SPREADSHEET_EXTENSIONS

EXCLUDED_DIR_NAMES = {"page_images"}  # pipeline-generated page images, not source documents
MIN_CONTENT_CHARS  = 50              # parsed docs with less text than this are quarantined
PAGE_IMAGES_DIR     = f"{VOLUME_PATH}/page_images"  # where ai_parse_document writes page images

RUN_ID = str(uuid.uuid4())
# Bump this when the parsing/chunking logic changes: files logged with a
# different version are automatically re-processed on the next run.
PIPELINE_VERSION = "v2-ai-prep-search"
print(f"Run {RUN_ID}")
print(f"volume={VOLUME_PATH} | gold={GOLD_TABLE} | log={LOG_TABLE} | index={INDEX_NAME}")
print(f"chunk_size={CHUNK_SIZE} | chunk_overlap={CHUNK_OVERLAP} | supported={sorted(SUPPORTED_EXTENSIONS)}")

Run 35e98154-46b2-42eb-9cb7-26fbb0a66040
volume=/Volumes/agentic_ai/01_bronze/raw_documents | gold=agentic_ai.03_gold.final_chunk | log=agentic_ai.01_bronze.document_processing_log | index=agentic_ai.03_gold.document_index
chunk_size=4000 | chunk_overlap=400 | supported=['doc', 'docx', 'jpeg', 'jpg', 'md', 'pdf', 'png', 'ppt', 'pptx', 'tif', 'tiff', 'txt', 'xls', 'xlsx']


In [0]:
# ------------------------------------------------------------------
# 2. STATE TABLE — one row per source file, updated every run.
#    A file is skipped on re-runs when its status is 'processed'
#    and its size + mtime are unchanged.
# ------------------------------------------------------------------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
  file_path      STRING   COMMENT 'Full path of the file in the UC volume',
  file_name      STRING   COMMENT 'Base file name',
  file_size      BIGINT   COMMENT 'Size in bytes at last processing',
  file_mtime_ms  BIGINT   COMMENT 'Modification time (epoch ms) at last processing',
  file_extension STRING   COMMENT 'Lower-case extension without the dot',
  status         STRING   COMMENT 'processed | failed | skipped (quarantined)',
  error_message  STRING   COMMENT 'Reason for failure or quarantine',
  chunks_written INT      COMMENT 'Chunks upserted for this file in the last run',
  run_id         STRING   COMMENT 'Id of the run that last touched this file',
  pipeline_version STRING COMMENT 'Version of the parsing/chunking logic applied',
  processed_at   TIMESTAMP COMMENT 'When this row was last updated'
)""")
# evolve a state table created by an earlier pipeline version
# (ALTER TABLE ... ADD COLUMN IF NOT EXISTS is not supported on all compute)
cols = [r["col_name"] for r in spark.sql(f"DESCRIBE TABLE {LOG_TABLE}").collect()]
if "pipeline_version" not in cols:
    spark.sql(f"ALTER TABLE {LOG_TABLE} ADD COLUMN pipeline_version STRING")
print(f"State table ready: {LOG_TABLE}")

State table ready: agentic_ai.01_bronze.document_processing_log


In [0]:
# ------------------------------------------------------------------
# 3. INCREMENTAL DISCOVERY — walk the volume once, keep only files
#    that are new, changed (size/mtime), or previously failed.
#    Unchanged, successfully processed files are skipped.
# ------------------------------------------------------------------
import os
from pyspark.sql.types import StructType, StructField, StringType, LongType

FILE_SCHEMA = StructType([
    StructField("file_path", StringType()),
    StructField("file_name", StringType()),
    StructField("file_size", LongType()),
    StructField("file_mtime_ms", LongType()),
    StructField("file_extension", StringType()),
])

discovered = []
for dirpath, dirnames, filenames in os.walk(VOLUME_PATH):
    # prune pipeline-internal / hidden directories in place
    dirnames[:] = [d for d in dirnames if d not in EXCLUDED_DIR_NAMES and not d.startswith(".")]
    for fn in sorted(filenames):
        if fn.startswith(".") or fn.startswith("~$") or fn.startswith("._"):
            continue  # hidden / office lock / macOS resource-fork files
        p = os.path.join(dirpath, fn)
        st = os.stat(p)
        discovered.append((
            p,
            fn,
            st.st_size,
            int(st.st_mtime * 1000),
            fn.rsplit(".", 1)[1].lower() if "." in fn else "",
        ))

disc_df = spark.createDataFrame(discovered, FILE_SCHEMA) if discovered else spark.createDataFrame([], FILE_SCHEMA)
disc_df.createOrReplaceTempView("discovered_files")

# New / changed / previously-failed files only:
#   - never seen before (no log row)
#   - size or mtime differ from the logged values (re-uploaded/edited)
#   - last attempt failed -> retry
# Unchanged quarantined files stay skipped; unchanged processed files are skipped.
candidates_df = spark.sql(f"""
    SELECT d.*
    FROM discovered_files d
    LEFT JOIN {LOG_TABLE} l ON l.file_path = d.file_path
    WHERE l.file_path IS NULL
       OR l.file_size      IS DISTINCT FROM d.file_size
       OR l.file_mtime_ms  IS DISTINCT FROM d.file_mtime_ms
       OR l.status = 'failed'
       OR l.pipeline_version IS DISTINCT FROM '{PIPELINE_VERSION}'  -- chunking logic changed: re-process
""")

candidate_rows = candidates_df.collect()
n_discovered     = len(discovered)
n_new_or_changed = len(candidate_rows)
print(f"Discovered {n_discovered} file(s) in {VOLUME_PATH}; {n_new_or_changed} new/changed/failed-to-retry.")
display(candidates_df)

Discovered 1 file(s) in /Volumes/agentic_ai/01_bronze/raw_documents; 1 new/changed/failed-to-retry.


file_path,file_name,file_size,file_mtime_ms,file_extension
/Volumes/agentic_ai/01_bronze/raw_documents/last_purchased_items_detailed_report.pdf,last_purchased_items_detailed_report.pdf,131728,1790788379000,pdf


In [0]:
# ------------------------------------------------------------------
# 4. PARSE -> VALIDATE -> CHUNK with per-file error isolation:
#    one bad file never fails the run; its status + error message
#    are recorded in the state table and the loop continues.
#    AI chunking: ai_parse_document -> ai_prep_search produces
#    semantic, context-enriched chunks (chunk_to_embed prepends the
#    document title / section headers for better retrieval).
#    TXT / XLS(X) files (not supported by the AI functions) fall
#    back to a simple fixed-size chunker.
# ------------------------------------------------------------------
import hashlib, json

class Quarantined(Exception):
    """File is intentionally skipped for a data-quality reason."""

def md5_hex(s: str) -> str:
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def sql_quote(s: str) -> str:
    return s.replace("\\", "\\\\").replace("'", "\\'")

def chunk_with_ai_functions(file_path: str, file_name: str) -> list:
    """ai_parse_document -> ai_prep_search chain: the AI functions parse
    the document and create semantic, context-enriched chunks. Returns
    chunk dicts matching the gold table schema."""
    rows = spark.sql(f"""
        WITH parsed_documents AS (
            SELECT ai_parse_document(
                       content,
                       MAP('version', '2.0',
                           'imageOutputPath', '{sql_quote(PAGE_IMAGES_DIR)}')) AS parsed
            FROM READ_FILES('{sql_quote(file_path)}', format => 'binaryFile')
        ), prepped_documents AS (
            SELECT parsed, ai_prep_search(parsed) AS result
            FROM parsed_documents
        )
        SELECT to_json(parsed:error_status) AS parse_errors,
               to_json(result)              AS result_json
        FROM prepped_documents
    """).collect()
    if not rows:
        raise Quarantined("READ_FILES returned no row (empty or unreadable file)")
    parse_errors = json.loads(rows[0]["parse_errors"] or "null")
    if parse_errors:
        raise Exception(f"ai_parse_document page errors: {json.dumps(parse_errors)[:500]}")
    result = json.loads(rows[0]["result_json"] or "null")
    if not result:
        raise Quarantined("ai_prep_search returned no result (empty or unreadable file)")
    if result.get("error_status"):
        raise Exception(f"ai_prep_search errors: {json.dumps(result['error_status'])[:500]}")

    contents = (result.get("document") or {}).get("contents") or []
    file_hash = md5_hex(file_path)
    chunks = []
    for i, c in enumerate(contents):
        page_uris = []
        for p in c.get("pages") or []:
            uri = p.get("image_uri")
            if uri and uri not in page_uris:
                page_uris.append(uri)
        chunks.append({
            "file_name": file_name,
            # deterministic id (hash of source path + chunk position) so
            # re-processing a file updates rows instead of duplicating them
            "chunk_id": f"{file_hash}_{c.get('chunk_position', i)}",
            "chunk_to_embed": c.get("chunk_to_embed") or "",
            "chunk_to_retrieve": c.get("chunk_to_retrieve") or "",
            "page_uris": page_uris,
        })
    return chunks

def parse_text_file(file_path: str) -> list:
    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        return [(f.read(), None)]  # (text, page_id)

def parse_spreadsheet(file_path: str) -> list:
    import pandas as pd
    segments = []
    sheets = pd.read_excel(file_path, sheet_name=None, header=None)
    for sheet_name, df in sheets.items():
        lines = [f"[Sheet: {sheet_name}]"]
        for row in df.itertuples(index=False):
            cells = ["" if pd.isna(v) else str(v) for v in row]
            lines.append(" | ".join(cells).rstrip(" |"))
        segments.append(("\n".join(lines), None))
    return segments

def make_chunks(file_path: str, file_name: str, segments: list, pages: list) -> list:
    """Fixed-size chunking with overlap; deterministic chunk_ids so
    re-processing a file updates rows instead of duplicating them."""

    def page_uris_for(page_ids):
        uris = []
        for pid in sorted(p for p in page_ids if p is not None):
            if isinstance(pid, int) and 0 <= pid < len(pages):
                uri = (pages[pid] or {}).get("image_uri")
                if uri and uri not in uris:
                    uris.append(uri)
        return uris

    # split any single segment longer than CHUNK_SIZE into fixed-size pieces
    normalized = []
    for text, pid in segments:
        if len(text) <= CHUNK_SIZE:
            normalized.append((text, pid))
        else:
            step = CHUNK_SIZE - CHUNK_OVERLAP
            offset = 0
            while offset < len(text):
                normalized.append((text[offset:offset + CHUNK_SIZE], pid))
                offset += step

    # `cur` holds (text, page_id) tuples so the overlap loop below can
    # iterate them; the chunk text is joined only at emit time.
    chunks, cur, cur_len, cur_pages = [], [], 0, set()

    def emit():
        text = "\n\n".join(t for t, _ in cur)
        chunks.append({
            "file_name": file_name,
            "chunk_id": None,  # assigned after the loop
            "chunk_to_embed": text,
            "chunk_to_retrieve": text,
            "page_uris": page_uris_for(cur_pages),
        })

    for text, pid in normalized:
        if cur and cur_len + len(text) + 2 > CHUNK_SIZE:
            emit()
            # carry a tail of the previous chunk into the next as overlap
            tail, tail_len, tail_pages = [], 0, set()
            for seg in reversed(cur):
                t, p = seg
                if tail_len + len(t) + 2 > CHUNK_OVERLAP:
                    break
                tail.insert(0, seg)
                tail_len += len(t) + 2
                tail_pages.add(p)
            cur, cur_len, cur_pages = tail, tail_len, tail_pages
        cur.append((text, pid))
        cur_len += len(text) + 2
        cur_pages.add(pid)
    if cur:
        emit()

    file_hash = md5_hex(file_path)
    for i, c in enumerate(chunks):
        c["chunk_id"] = f"{file_hash}_{i}"
    return chunks

run_chunks, file_results = [], []

for r in candidate_rows:
    fp, fn = r["file_path"], r["file_name"]
    result = {
        "file_path": fp, "file_name": fn, "file_size": r["file_size"],
        "file_mtime_ms": r["file_mtime_ms"], "file_extension": r["file_extension"],
        "status": "processed", "error_message": None, "chunks_written": 0,
        "run_id": RUN_ID, "pipeline_version": PIPELINE_VERSION,
    }
    try:
        ext = r["file_extension"]
        if ext not in SUPPORTED_EXTENSIONS:
            raise Quarantined(f"Unsupported file type '{ext or '(none)'}'; supported: {sorted(SUPPORTED_EXTENSIONS)}")

        if ext in AI_PARSE_EXTENSIONS:
            # AI chunking: ai_parse_document -> ai_prep_search
            file_chunks = chunk_with_ai_functions(fp, fn)
        elif ext in TEXT_EXTENSIONS:
            file_chunks = make_chunks(fp, fn, parse_text_file(fp), [])
        else:  # spreadsheet
            file_chunks = make_chunks(fp, fn, parse_spreadsheet(fp), [])

        total_chars = sum(len(c["chunk_to_retrieve"]) for c in file_chunks)
        if not file_chunks or total_chars < MIN_CONTENT_CHARS:
            raise Quarantined(f"Chunked content too short ({total_chars} chars < {MIN_CONTENT_CHARS}); likely empty or corrupt document")

        run_chunks.extend(file_chunks)
        result["chunks_written"] = len(file_chunks)
        print(f"  processed {fn}: {len(file_chunks)} chunks from {total_chars} chars")
    except Quarantined as q:
        result.update(status="skipped", error_message=str(q))
        print(f"  skipped   {fn}: {q}")
    except Exception as e:
        result.update(status="failed", error_message=f"{type(e).__name__}: {str(e)[:500]}")
        print(f"  FAILED    {fn}: {type(e).__name__}: {str(e)[:200]}")
    file_results.append(result)

n_processed_now = sum(1 for r in file_results if r["status"] == "processed")
print(f"\nParsed {n_processed_now} file(s) -> {len(run_chunks)} chunk(s) ready for the gold table.")

  processed last_purchased_items_detailed_report.pdf: 5 chunks from 17319 chars

Parsed 1 file(s) -> 5 chunk(s) ready for the gold table.


In [0]:
# ------------------------------------------------------------------
# 5. WRITE TO GOLD — idempotent MERGE keyed on chunk_id (updates
#    re-processed chunks instead of duplicating), then delete stale
#    chunks of re-uploaded files whose chunk count shrank.
#    The existing schema is preserved exactly: file_name, chunk_id,
#    chunk_to_embed, chunk_to_retrieve, page_uris.
# ------------------------------------------------------------------
from pyspark.sql.types import ArrayType, IntegerType

chunks_written_total = 0
if run_chunks:
    CHUNKS_SCHEMA = StructType([
        StructField("file_name", StringType()),
        StructField("chunk_id", StringType()),
        StructField("chunk_to_embed", StringType()),
        StructField("chunk_to_retrieve", StringType()),
        StructField("page_uris", ArrayType(StringType())),
    ])
    chunks_df = spark.createDataFrame(run_chunks, schema=CHUNKS_SCHEMA)
    chunks_df.createOrReplaceTempView("run_chunks")

    merge_metrics = spark.sql(f"""
        MERGE INTO {GOLD_TABLE} AS g
        USING run_chunks AS s
        ON g.chunk_id = s.chunk_id
        WHEN MATCHED THEN UPDATE SET
            g.file_name        = s.file_name,
            g.chunk_to_embed   = s.chunk_to_embed,
            g.chunk_to_retrieve = s.chunk_to_retrieve,
            g.page_uris        = s.page_uris
        WHEN NOT MATCHED THEN INSERT
            (file_name, chunk_id, chunk_to_embed, chunk_to_retrieve, page_uris)
            VALUES (s.file_name, s.chunk_id, s.chunk_to_embed, s.chunk_to_retrieve, s.page_uris)
    """).collect()
    try:
        print("MERGE metrics:", [row.asDict() for row in merge_metrics])
    except Exception:
        pass

    # a re-uploaded file may now produce fewer chunks: remove its orphans
    delete_metrics = spark.sql(f"""
        DELETE FROM {GOLD_TABLE}
        WHERE file_name IN (SELECT DISTINCT file_name FROM run_chunks)
          AND chunk_id NOT IN (SELECT chunk_id FROM run_chunks)
    """).collect()
    try:
        print("DELETE metrics:", [row.asDict() for row in delete_metrics])
    except Exception:
        pass

    chunks_written_total = len(run_chunks)
    print(f"Upserted {chunks_written_total} chunk(s) into {GOLD_TABLE} (stale chunks of re-processed files removed).")
else:
    print("No new chunks to write.")

MERGE metrics: [{'num_affected_rows': 5, 'num_updated_rows': 0, 'num_deleted_rows': 0, 'num_inserted_rows': 5}]
DELETE metrics: [{'num_affected_rows': 6}]
Upserted 5 chunk(s) into agentic_ai.03_gold.final_chunk (stale chunks of re-processed files removed).


In [0]:
# ------------------------------------------------------------------
# 6. UPDATE STATE TABLE — recorded AFTER the gold write, so a crash
#    between the two simply re-processes the same files on the next
#    run (harmless: the MERGE on chunk_id is idempotent).
# ------------------------------------------------------------------
from pyspark.sql import functions as F

if file_results:
    LOG_UPDATE_SCHEMA = StructType([
        StructField("file_path", StringType()),
        StructField("file_name", StringType()),
        StructField("file_size", LongType()),
        StructField("file_mtime_ms", LongType()),
        StructField("file_extension", StringType()),
        StructField("status", StringType()),
        StructField("error_message", StringType()),
        StructField("chunks_written", IntegerType()),
        StructField("run_id", StringType()),
        StructField("pipeline_version", StringType()),
    ])
    updates = [{k: v for k, v in r.items() if k != "processed_at"} for r in file_results]
    log_df = (spark.createDataFrame(updates, schema=LOG_UPDATE_SCHEMA)
              .withColumn("processed_at", F.current_timestamp()))
    log_df.createOrReplaceTempView("run_log_updates")

    spark.sql(f"""
        MERGE INTO {LOG_TABLE} AS l
        USING run_log_updates AS u
        ON l.file_path = u.file_path
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Recorded {len(file_results)} file result(s) in {LOG_TABLE}.")
else:
    print("No state-table updates (no candidate files this run).")

Recorded 1 file result(s) in agentic_ai.01_bronze.document_processing_log.


In [0]:
# ------------------------------------------------------------------
# 7. TRIGGER AI SEARCH INDEX SYNC — embeddings are computed by the
#    managed DELTA_SYNC index (embedding column chunk_to_embed,
#    model databricks-qwen3-embedding-0-6b). We never compute
#    embeddings ourselves; for a TRIGGERED pipeline we just ask the
#    index to pick up the new rows.
# ------------------------------------------------------------------
from databricks.sdk import WorkspaceClient

index_sync_triggered = False
if chunks_written_total > 0:
    w = WorkspaceClient()  # uses the notebook's native auth, no tokens
    response = w.vector_search_indexes.sync_index(index_name=INDEX_NAME)
    index_sync_triggered = True
    print(f"Index sync triggered for {INDEX_NAME}.")
    print(response)
else:
    print("No new chunks — index sync skipped (nothing to embed).")

Index sync triggered for agentic_ai.03_gold.document_index.
None


In [0]:
# ------------------------------------------------------------------
# 8. RUN SUMMARY
# ------------------------------------------------------------------
n_processed = sum(1 for r in file_results if r["status"] == "processed")
n_failed    = sum(1 for r in file_results if r["status"] == "failed")
n_skipped   = sum(1 for r in file_results if r["status"] == "skipped")
n_unchanged = n_discovered - n_new_or_changed

print("=" * 62)
print(f"RUN SUMMARY — {RUN_ID}")
print("=" * 62)
print(f"Files discovered            : {n_discovered}")
print(f"  already up to date        : {n_unchanged}")
print(f"  processed this run        : {n_processed}")
print(f"  quarantined (skipped)     : {n_skipped}")
print(f"  failed                    : {n_failed}")
print(f"Chunks upserted to gold     : {chunks_written_total}")
print(f"AI Search index sync        : {'triggered' if index_sync_triggered else 'skipped (no changes)'}")

to_investigate = [r for r in file_results if r["status"] in ("failed", "skipped")]
if to_investigate:
    print("\nFiles to investigate:")
    for r in to_investigate:
        print(f"  [{r['status']:7s}] {r['file_name']}: {r['error_message']}")
else:
    print("\nNo files flagged for investigation.")

gold_stats = spark.sql(f"""
    SELECT count(*) AS total_chunks, count(DISTINCT file_name) AS total_files
    FROM {GOLD_TABLE}
""").collect()[0]
print(f"\nGold table {GOLD_TABLE} now holds {gold_stats['total_chunks']} chunk(s) across {gold_stats['total_files']} file(s).")
display(spark.sql(f"SELECT * FROM {LOG_TABLE} ORDER BY processed_at DESC"))

RUN SUMMARY — e912c9c3-cebd-45d9-b5c0-10b2a0560210
Files discovered            : 1
  already up to date        : 0
  processed this run        : 0
  quarantined (skipped)     : 0
  failed                    : 1
Chunks upserted to gold     : 0
AI Search index sync        : skipped (no changes)

Files to investigate:
  [failed ] last_purchased_items_detailed_report.pdf: ValueError: too many values to unpack (expected 2)

Gold table agentic_ai.03_gold.final_chunk now holds 6 chunk(s) across 1 file(s).


file_path,file_name,file_size,file_mtime_ms,file_extension,status,error_message,chunks_written,run_id,processed_at
/Volumes/agentic_ai/01_bronze/raw_documents/last_purchased_items_detailed_report.pdf,last_purchased_items_detailed_report.pdf,131728,1790788379000,pdf,failed,ValueError: too many values to unpack (expected 2),0,e912c9c3-cebd-45d9-b5c0-10b2a0560210,2026-11-04T17:25:28.106Z
